In [26]:
import glob
import seaborn as sns
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("max_colwidth", 100)
sns.set(rc={"figure.figsize": (20, 12)})

In [40]:
files_list = glob.glob("../resultsv2/dtm/trump/*.json")
dfs = []
for f in files_list:
    embedding_retrieval_type = "_".join(f.split("_")[4:-1]) + "_pooling"
    df = pd.read_json(f)
    df["embedding_retrieval_type"] = embedding_retrieval_type
    dfs.append(df)

In [41]:
full_results = pd.concat(dfs)

In [42]:
keys_to_extract = ["embedding_model", "nr_topics", "min_topic_size"]
for key in keys_to_extract:
    full_results[key] = full_results["Params"].apply(lambda x: x.get(key, None))
full_results.drop("Params", axis=1, inplace=True)

In [43]:
npmis = []
diversities = []
import numpy as np

for _, row in full_results.iterrows():
    npmi = round(np.mean([row["Scores"][timestamp]["npmi"] for timestamp in row["Scores"].keys()]), 3)
    diversity = round(np.mean([row["Scores"][timestamp]["diversity"] for timestamp in row["Scores"].keys()]), 3)
    npmis.append(npmi)
    diversities.append(diversity)

In [44]:
full_results["npmi"] = npmis
full_results["diversity"] = diversities

In [45]:
results_grouped = full_results.groupby(
    [full_results["embedding_retrieval_type"], full_results["nr_topics"]], as_index=False
)[["nr_topics", "npmi", "diversity", "Computation Time"]].mean()

In [46]:
results_grouped["npmi"] = np.round(results_grouped["npmi"], 3)
results_grouped["diversity"] = np.round(results_grouped["diversity"], 3)

In [47]:
def rename_embedding_type(name):
    # Split the name into parts
    parts = name.split("_")

    # Define the new names for each part
    new_name_parts = {
        "concat": "Concat",
        "sum": "Sum",
        "last": "Last",
        "second": "Second",
        "all": "All",
        "four": "Four",
        "layers": "Layers",
        "layer": "Layer",
        "output": "Output",
        "hidden": "Hidden",
    }

    # Determine the pooling type and place it in parentheses
    pooling = parts[-2].upper() if parts[-2] == "cls" else parts[-2].capitalize()
    pooling_type = f"({pooling} Pooling)"

    # Check for 'embedding_layer' to include "Embedding Layer" in the new name
    if "_".join(parts[:2]) == "embedding_layer":
        new_name = "Embedding Layer " + " ".join(
            new_name_parts.get(part, part.capitalize()) for part in parts[2:-2]
        )
    else:
        new_name = " ".join(new_name_parts.get(part, part.capitalize()) for part in parts[1:-2])

    # Add the pooling type
    new_name += " " + pooling_type

    # Remove extra spaces and return
    return " ".join(new_name.split())

In [48]:
results_grouped["embedding_retrieval_type"] = [rename_embedding_type(name) for name in results_grouped["embedding_retrieval_type"]]

In [49]:
results_sorted_npmi = (
    results_grouped.groupby("embedding_retrieval_type")
    .mean("npmi")
    .reset_index()
    .sort_values("npmi", ascending=False)
)
top_categories_npmi = results_sorted_npmi.head(5)["embedding_retrieval_type"].tolist()

In [51]:
results_sorted_npmi.sort_values("npmi").drop(["nr_topics", "Computation Time"], axis=1)

,embedding_retrieval_type,npmi,diversity
3,Embedding Layer Output (CLS Pooling),-0.099,0.945
9,Second Last Layer (CLS Pooling),0.050,0.812
0,Concat Last Four Layers (CLS Pooling),0.073,0.863
17,Sum Last Four Layers (Mean Pooling),0.074,0.872
6,Last Hidden Layer (CLS Pooling),0.076,0.877
15,Sum Last Four Layers (CLS Pooling),0.076,0.854
12,Sum All Layers (CLS Pooling),0.077,0.852
2,Concat Last Four Layers (Mean Pooling),0.078,0.863
8,Last Hidden Layer (Mean Pooling),0.082,0.875
11,Second Last Layer (Mean Pooling),0.084,0.859
